# Part D: Confidence and Calibration Analysis

**Owner:** Mythri S  
**Project:** PW26_VK_06

**What this notebook does:**
- Computes ECE (Expected Calibration Error) for each of the 4 encoder-swap conditions
- Computes confident-wrong rate (confidence > 0.9 and prediction is wrong)
- Plots reliability diagrams
- Saves `confidence_results.csv` and `reliability_diagram.png`

**Input:** `results/swap/predictions.csv`  
**Output:** `results/confidence/confidence_results.csv` + `results/confidence/reliability_diagram.png`

## Cell 1: Clone repo (run on Colab)

In [ ]:
import subprocess, os

REPO_URL = 'https://github.com/Shikshith05/encoder-forgetting-signal.git'
REPO_DIR = '/content/encoder-forgetting-signal'

if not os.path.isdir(REPO_DIR):
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
else:
    print('Repo already exists, pulling latest...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Cell 2: Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

PREDICTIONS_PATH = 'results/swap/predictions.csv'
OUTPUT_DIR       = 'results/confidence'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup complete.')
print(f'Predictions file exists: {os.path.isfile(PREDICTIONS_PATH)}')

## Cell 3: Load Predictions

In [ ]:
df = pd.read_csv(PREDICTIONS_PATH)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Pairs:', df['pair'].unique())
print('Hybrids:', df['hybrid'].unique())
df.head(6)

## Cell 4: ECE and Confident-Wrong Rate Functions

In [ ]:
def compute_ece(confs, corrects, n_bins=15):
    confs    = np.array(confs)
    corrects = np.array(corrects, dtype=float)
    n        = len(confs)
    bins     = np.linspace(0, 1, n_bins + 1)
    ece      = 0.0
    bin_data = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confs >= lo) & (confs <= hi) if i == n_bins-1 else (confs >= lo) & (confs < hi)
        count = mask.sum()
        if count == 0:
            continue
        avg_conf = confs[mask].mean()
        avg_acc  = corrects[mask].mean()
        ece += (count / n) * abs(avg_conf - avg_acc)
        bin_data.append(((lo+hi)/2, avg_conf, avg_acc, count))
    return ece, bin_data

def compute_confident_wrong_rate(confs, corrects, threshold=0.9):
    confs    = np.array(confs)
    corrects = np.array(corrects, dtype=float)
    mask = confs > threshold
    if mask.sum() == 0:
        return 0.0, 0
    rate = ((1 - corrects) * mask).sum() / mask.sum() * 100
    return rate, int(mask.sum())

print('Functions defined.')

## Cell 5: Compute Metrics for All Pairs

In [ ]:
results = []
bin_data_store = {}

for pair in df['pair'].unique():
    for hybrid in ['pp', 'pf', 'fp', 'ff']:
        subset = df[(df['pair'] == pair) & (df['hybrid'] == hybrid)]
        if len(subset) == 0:
            continue
        confs    = subset['conf'].values
        corrects = (subset['true'] == subset['pred']).values
        ece, bin_data = compute_ece(confs, corrects)
        cw_rate, n_high_conf = compute_confident_wrong_rate(confs, corrects)
        accuracy = corrects.mean() * 100
        results.append({'pair': pair, 'swap_condition': hybrid, 'n_images': len(subset),
                        'accuracy_pct': round(accuracy, 2), 'ece': round(ece, 4),
                        'confident_wrong_rate_pct': round(cw_rate, 2), 'n_high_conf': n_high_conf})
        bin_data_store[(pair, hybrid)] = bin_data

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## Cell 6: Reliability Diagram

In [ ]:
MAIN_PAIR = 'Pets->EuroSAT (lr 3e-5)'
if MAIN_PAIR not in df['pair'].unique():
    MAIN_PAIR = 'Pets->EuroSAT'
print(f'Using: {MAIN_PAIR}')

colors = {'pp': '#2ecc71', 'pf': '#3498db', 'fp': '#e67e22', 'ff': '#e74c3c'}
labels = {'pp': 'A_pp (both pretrained)', 'pf': 'A_pf (vision restored)',
          'fp': 'A_fp (text restored)', 'ff': 'A_ff (both fine-tuned)'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Reliability Diagram — {MAIN_PAIR}', fontsize=13, fontweight='bold')

for ax_idx, focus in enumerate([['pp', 'ff'], ['pf', 'fp']]):
    ax = axes[ax_idx]
    ax.plot([0,1],[0,1],'k--',linewidth=1,label='Perfect calibration',alpha=0.6)
    for hybrid in focus:
        key = (MAIN_PAIR, hybrid)
        if key not in bin_data_store or len(bin_data_store[key]) == 0:
            continue
        bdata = bin_data_store[key]
        avg_confs = [b[1] for b in bdata]
        avg_accs  = [b[2] for b in bdata]
        sizes = [max(30, b[3]/5) for b in bdata]
        ax.scatter(avg_confs, avg_accs, s=sizes, color=colors[hybrid], label=labels[hybrid], zorder=5, alpha=0.85)
        ax.plot(avg_confs, avg_accs, color=colors[hybrid], linewidth=1.5, alpha=0.7)
    ax.set_title('Before vs After' if ax_idx==0 else 'Encoder Restoration')
    ax.set_xlabel('Mean Confidence'); ax.set_ylabel('Accuracy')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reliability diagram.')

## Cell 7: Save CSV

In [ ]:
results_df.to_csv(f'{OUTPUT_DIR}/confidence_results.csv', index=False)
print(f'Saved {OUTPUT_DIR}/confidence_results.csv')
print(results_df.to_string(index=False))

## Cell 8: Summary

In [ ]:
print('ECE: how overconfident/underconfident the model is (lower = better calibrated)')
print('CW%: % of high-confidence (>0.9) predictions that are WRONG (our CLIP hallucination metric)')
print()
for pair in results_df['pair'].unique():
    sub = results_df[results_df['pair']==pair]
    print(f'Pair: {pair}')
    for _, r in sub.iterrows():
        print(f'  {r["swap_condition"]}  acc={r["accuracy_pct"]}%  ECE={r["ece"]}  CW%={r["confident_wrong_rate_pct"]}%')
    print()